Unsa Chaudhry

used ChatGPT to help with propositions

1\. Customers who have placed both online and in-store orders

INTERSECT between online vs in-store sets. 

Tables involved: Sales.SalesOrderHeader

In [ ]:
SELECT DISTINCT CustomerID
FROM Sales.SalesOrderHeader
WHERE OnlineOrderFlag = 1
INTERSECT
SELECT DISTINCT CustomerID
FROM Sales.SalesOrderHeader
WHERE OnlineOrderFlag = 0;

2\. New customers this year (ordered this year but not last year) 

EXCEPT between this-year and last-year customer sets. 

tables involved: Sales.SalesOrderHeader

In [ ]:
WITH ThisYear AS (
  SELECT DISTINCT CustomerID
  FROM Sales.SalesOrderHeader
  WHERE YEAR(OrderDate) = YEAR(GETDATE())
),
LastYear AS (
  SELECT DISTINCT CustomerID
  FROM Sales.SalesOrderHeader
  WHERE YEAR(OrderDate) = YEAR(DATEADD(year, -1, GETDATE()))
)
SELECT CustomerID FROM ThisYear
EXCEPT
SELECT CustomerID FROM LastYear;

3\. Vendor cities that do not appear in customer addresses 

EXCEPT between vendor-city set and customer-city set. 

tables involved: Purchasing.Vendor, Person.BusinessEntityAddress, Person.Address, Sales.Customer, Sales.Store

In [ ]:
WITH VendorCities AS (
  SELECT DISTINCT a.City
  FROM Purchasing.Vendor v
  JOIN Person.BusinessEntityAddress bea ON bea.BusinessEntityID = v.BusinessEntityID
  JOIN Person.Address a ON a.AddressID = bea.AddressID
),
CustomerCities AS (
  SELECT DISTINCT a.City
  FROM Sales.Customer c
  JOIN Person.BusinessEntityAddress bea ON bea.BusinessEntityID = c.PersonID
  JOIN Person.Address a ON a.AddressID = bea.AddressID
  WHERE c.PersonID IS NOT NULL
  UNION
  SELECT DISTINCT a.City
  FROM Sales.Customer c
  JOIN Sales.Store s ON s.BusinessEntityID = c.StoreID
  JOIN Person.BusinessEntityAddress bea ON bea.BusinessEntityID = s.BusinessEntityID
  JOIN Person.Address a ON a.AddressID = bea.AddressID
  WHERE c.StoreID IS NOT NULL
)
SELECT City FROM VendorCities
EXCEPT
SELECT City FROM CustomerCities;

4\. People who are both customers and employees 

INTERSECT between customer person-ids and employee ids. 

tables involved: Sales.Customer, HumanResources.Employee

In [ ]:
SELECT PersonID AS BusinessEntityID
FROM Sales.Customer
WHERE PersonID IS NOT NULL
INTERSECT
SELECT BusinessEntityID
FROM HumanResources.Employee;

5\. Unified list of individual customers, store customers, and vendors

UNION to combine three role sets. 

tables involved: Sales.Customer, Person.Person, Sales.Store, Purchasing.Vendor

In [ ]:
SELECT p.FirstName + p.LastName AS Name, 'Individual Customer' AS Role
FROM Sales.Customer c
JOIN Person.Person p ON p.BusinessEntityID = c.PersonID
WHERE c.PersonID IS NOT NULL
UNION
SELECT s.Name, 'Store Customer' AS Role
FROM Sales.Customer c
JOIN Sales.Store s ON s.BusinessEntityID = c.StoreID
WHERE c.StoreID IS NOT NULL
UNION
SELECT v.Name, 'Vendor' AS Role
FROM Purchasing.Vendor v;


6\. Products sold but never purchased from vendors 

EXCEPT between sold-product set and purchased-product set. 

tables involved: Sales.SalesOrderDetail, Purchasing.PurchaseOrderDetail, Production.Product

In [ ]:
WITH Sold AS (
  SELECT DISTINCT sod.ProductID
  FROM Sales.SalesOrderDetail sod
),
Purchased AS (
  SELECT DISTINCT pod.ProductID
  FROM Purchasing.PurchaseOrderDetail pod
)
SELECT p.ProductID, p.Name
FROM Sold s
JOIN Production.Product p ON p.ProductID = s.ProductID
EXCEPT
SELECT p2.ProductID, p2.Name
FROM Purchased pr
JOIN Production.Product p2 ON p2.ProductID = pr.ProductID;


7\. Products purchased from vendors but never sold to customers 

EXCEPT between purchased-product set and sold-product set. 

tables involved: Purchasing.PurchaseOrderDetail, Sales.SalesOrderDetail, Production.Product

In [ ]:
WITH Purchased AS (
  SELECT DISTINCT pod.ProductID
  FROM Purchasing.PurchaseOrderDetail pod
),
Sold AS (
  SELECT DISTINCT sod.ProductID
  FROM Sales.SalesOrderDetail sod
)
SELECT p.ProductID, p.Name
FROM Purchased pr
JOIN Production.Product p ON p.ProductID = pr.ProductID
EXCEPT
SELECT p2.ProductID, p2.Name
FROM Sold s
JOIN Production.Product p2 ON p2.ProductID = s.ProductID;

8\. Territories that have either customers or salespeople 

UNION to combine territory ids from both sets. 

tables involved: Sales.Customer, Sales.SalesPerson

In [ ]:
SELECT DISTINCT TerritoryID
FROM Sales.Customer
WHERE TerritoryID IS NOT NULL
UNION
SELECT DISTINCT TerritoryID
FROM Sales.SalesPerson
WHERE TerritoryID IS NOT NULL;

9\. Cities shared by both vendors and customers 

NTERSECT between vendor-city set and customer-city set. 

tables involved: Purchasing.Vendor, Sales.Customer, Sales.Store, Person.BusinessEntityAddress, Person.Address

In [ ]:
WITH VendorCities AS (
  SELECT DISTINCT a.City
  FROM Purchasing.Vendor v
  JOIN Person.BusinessEntityAddress bea ON bea.BusinessEntityID = v.BusinessEntityID
  JOIN Person.Address a ON a.AddressID = bea.AddressID
),
CustomerCities AS (
  SELECT DISTINCT a.City
  FROM Sales.Customer c
  JOIN Person.BusinessEntityAddress bea ON bea.BusinessEntityID = c.PersonID
  JOIN Person.Address a ON a.AddressID = bea.AddressID
  WHERE c.PersonID IS NOT NULL
  UNION
  SELECT DISTINCT a.City
  FROM Sales.Customer c
  JOIN Sales.Store s ON s.BusinessEntityID = c.StoreID
  JOIN Person.BusinessEntityAddress bea ON bea.BusinessEntityID = s.BusinessEntityID
  JOIN Person.Address a ON a.AddressID = bea.AddressID
  WHERE c.StoreID IS NOT NULL
)
SELECT City FROM VendorCities
INTERSECT
SELECT City FROM CustomerCities;

10\. Customers who bought from two different subcategories 

INTERSECT between customer sets for two subcategories 

tables involved: Sales.SalesOrderHeader, Sales.SalesOrderDetail, Production.Product, Production.ProductSubcategory

In [ ]:
WITH CustRoadBikes AS (
  SELECT DISTINCT soh.CustomerID
  FROM Sales.SalesOrderHeader soh
  JOIN Sales.SalesOrderDetail sod ON sod.SalesOrderID = soh.SalesOrderID
  JOIN Production.Product p ON p.ProductID = sod.ProductID
  JOIN Production.ProductSubcategory ps ON ps.ProductSubcategoryID = p.ProductSubcategoryID
  WHERE ps.Name = 'Road Bikes'
),
CustHelmets AS (
  SELECT DISTINCT soh.CustomerID
  FROM Sales.SalesOrderHeader soh
  JOIN Sales.SalesOrderDetail sod ON sod.SalesOrderID = soh.SalesOrderID
  JOIN Production.Product p ON p.ProductID = sod.ProductID
  JOIN Production.ProductSubcategory ps ON ps.ProductSubcategoryID = p.ProductSubcategoryID
  WHERE ps.Name = 'Helmets'
)
SELECT CustomerID FROM CustRoadBikes
INTERSECT
SELECT CustomerID FROM CustHelmets;